# 处理数据

In [13]:
from torch.utils.data import Dataset
import json

class AFQMC(Dataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)
    
    def load_data(self, data_file):
        data = {}
        with open(data_file, 'rt') as f:
            for idx, line in enumerate(f):
                data[idx] = json.loads(line.strip())
        return data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

迭代生成

In [40]:
from typing import Iterator
from torch.utils.data import IterableDataset
class IterAFQMC(IterableDataset):
    def __init__(self, data_file) -> None:
        super().__init__()
        self.data_file = data_file

    def __iter__(self) -> Iterator:
        with open(self.data_file, 'rt') as f:
            for idx, line in enumerate(f):
                yield json.loads(line.strip())

In [41]:
da_set = IterAFQMC('../data/afqmc_public/train.json')
next(iter(da_set))

{'sentence1': '蚂蚁借呗等额还款可以换成先息后本吗', 'sentence2': '借呗有先息到期还本吗', 'label': '0'}

In [14]:
dataset = AFQMC('../data/afqmc_public/train.json')
dataset.data[0]

{'sentence1': '蚂蚁借呗等额还款可以换成先息后本吗', 'sentence2': '借呗有先息到期还本吗', 'label': '0'}

In [6]:
import torch
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

In [ ]:
check_point = 'bert-base-chinese'
tokenizer = AutoTokenizer.from_pretrained(check_point)

In [15]:
def collate_fn(samples):
    seq1, seq2 = [], []
    labels = []
    for sample in samples:
        seq1.append(sample['sentence1'])
        seq2.append(sample['sentence2'])
        labels.append(int(sample['label']))
    
    X = tokenizer(
        seq1,
        seq2,
        padding=True,
        truncation=True,
        return_tensors='pt'
    )

    # label
    y = torch.tensor(labels)

    return X, y

data_loader = DataLoader(dataset, batch_size=4, collate_fn=collate_fn, shuffle=True)

In [46]:
iter_dataloader = DataLoader(da_set, batch_size=4, collate_fn=collate_fn)

In [31]:
data = next(iter(data_loader))[0]

In [43]:
from torch import nn
from transformers import BertPreTrainedModel, BertModel, AutoConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'

class MyModel(BertPreTrainedModel):
    def __init__(self, config: AutoConfig):
        super().__init__(config)

        self.bert = BertModel(config, add_pooling_layer=False)
        self.drop = nn.Dropout(config.hidden_dropout_prob) # 0.1 是个值
        self.classifier = nn.Linear(768, 2)
        self.post_init()

    def forward(self, x):
        # x: input_ids: tokens -> [batch, seq_length]}
        outs = self.bert(**x)
        cls_vecs = outs.last_hidden_state[:, 0, :]
        cls_vecs = self.drop(cls_vecs)

        logits = self.classifier(cls_vecs)

        return logits

m_config = AutoConfig.from_pretrained(check_point)

model = MyModel.from_pretrained(check_point, config=m_config).to(device)

/data/home/zhangchangtian/software/miniforge3/envs/algo_learn/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
loading configuration file config.json from cache at /data/home/zhangchangtian/.cache/huggingface/hub/models--bert-base-chinese/snapshots/c30a6ed22ab4564dc1e3b2ecbf6e766b0611a33f/config.json
Model config BertConfig {
  "_name_or_path": "bert-base-chinese",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "directionality": "bidi",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_la

In [50]:
model(data.to(device))

tensor([[-0.0445, -0.1292],
        [-0.4475, -0.2550],
        [-0.1326, -0.1530],
        [-0.1266, -0.2586]], device='cuda:0', grad_fn=<AddmmBackward0>)

# Train Loop

In [51]:
# Train
from transformers import AdamW, get_scheduler
from tqdm.auto import tqdm

def train_loop(dataloader, model, loss_fn, optimizer, lr_scheduler, epoch, total_loss):
    progress_bar = tqdm(range(len(dataloader)))
    progress_bar.set_description(f'loss: {0:>7f}')
    finish_step_num = (epoch-1)*len(dataloader)
    
    model.train()
    for step, (X, y) in enumerate(dataloader, start=1):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        progress_bar.set_description(f'loss: {total_loss/(finish_step_num + step):>7f}')
        progress_bar.update(1)
    return total_loss

learning_rate = 1e-5
epoch_num = 3

loss_fn = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=learning_rate)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=epoch_num*len(data_loader),
)

total_loss = 0.
for t in range(epoch_num):
    print(f"Epoch {t+1}/{epoch_num}\n-------------------------------")
    total_loss = train_loop(data_loader, model, loss_fn, optimizer, lr_scheduler, t+1, total_loss)
    # test_loop(valid_dataloader, model, mode='Valid')
print("Done!")

Epoch 1/3
-------------------------------


/data/home/zhangchangtian/software/miniforge3/envs/algo_learn/lib/python3.9/site-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


  0%|          | 0/8584 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [2]:
import torch
torch.cuda.empty_cache()